---
## 🌐 Multi-Network GNN — All 176 Web of Life Pollination Networks

This notebook extends the single-network GNN novelty to **all 176 networks**.
Each network is processed independently (no cross-ecosystem merging).
Results are aggregated for comparative ecological analysis.

**Pipeline per network:**
```
CSV → bipartite graph → GNN embeddings → KMeans clustering → role assignments → save
```
**Then across all 176 results:**
```
role distributions → network-level stats → 6 publication-quality visualisations
```

### Cell 1 — Imports & global settings

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os, glob, random, json, time
import numpy as np
import pandas as pd
import networkx as nx
from networkx.algorithms import bipartite

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import from_networkx, negative_sampling
from torch_geometric.nn import SAGEConv
from sklearn.metrics import roc_auc_score

SEED   = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# ── Paths — EDIT THESE ──────────────────────────────────────────────────────
DATA_DIR = r"C:\Users\akash\Videos\Mtech_IIT\2nd sem\network science\project\ns\data"
OUT_DIR  = r"C:\Users\akash\Videos\Mtech_IIT\2nd sem\network science\project\ns\outputs\multi_network"
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "per_network"), exist_ok=True)

# Colour palette (consistent across all plots)
BG           = "#0D1117"
TXT          = "#E6EDF3"
GRID         = "#30363D"
ROLE_COLORS  = {
    "Generalist" : "#4CAF50",
    "Specialist"  : "#42A5F5",
    "Connector"   : "#FF7043",
    "Keystone"    : "#FFD700",
}
ALL_ROLES = ["Generalist", "Specialist", "Connector", "Keystone"]

def style_ax(ax, title):
    ax.set_facecolor(BG)
    ax.set_title(title, color=TXT, fontsize=9, fontweight="bold", pad=7)
    ax.tick_params(colors=TXT, labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID)
    ax.xaxis.label.set_color(TXT)
    ax.yaxis.label.set_color(TXT)

print("Imports OK")

ModuleNotFoundError: No module named 'torch_geometric'

### Cell 2 — Shared helper functions (graph, features, model)

In [ ]:
# ── Graph builder ─────────────────────────────────────────────────────────
def build_bipartite(matrix_df):
    G = nx.Graph()
    pl  = list(matrix_df.index)
    pol = list(matrix_df.columns)
    G.add_nodes_from(pl,  bipartite=0, kind="plant")
    G.add_nodes_from(pol, bipartite=1, kind="pollinator")
    for p in pl:
        for q in pol:
            w = matrix_df.loc[p, q]
            if w > 0:
                G.add_edge(p, q, weight=float(w))
    return G


# ── Node feature builder ───────────────────────────────────────────────────
def build_pyg_data(G):
    """6 structural features per node, standardised."""
    btw   = nx.betweenness_centrality(G, weight="weight", normalized=True)
    clust = nx.clustering(G)
    node_order = list(G.nodes())
    rows = []
    for n in node_order:
        deg  = float(G.degree(n))
        wdeg = float(sum(d.get("weight", 1.0) for _, _, d in G.edges(n, data=True)))
        side = float(G.nodes[n].get("bipartite", 0))
        rows.append([deg, wdeg, btw.get(n, 0.0), clust.get(n, 0.0), side, float(np.log1p(deg))])
    X_raw    = np.array(rows, dtype=np.float32)
    X_scaled = StandardScaler().fit_transform(X_raw).astype(np.float32)
    pyg      = from_networkx(G)
    pyg.x    = torch.tensor(X_scaled, dtype=torch.float)
    pyg.node_names = node_order
    return pyg, node_order, X_raw


# ── GNN model ─────────────────────────────────────────────────────────────
class GraphSAGEEncoder(nn.Module):
    def __init__(self, in_ch, hid, emb, drop=0.3):
        super().__init__()
        self.conv1    = SAGEConv(in_ch, hid)
        self.conv2    = SAGEConv(hid, hid)
        self.conv3    = SAGEConv(hid, emb)
        self.bn1      = nn.BatchNorm1d(hid)
        self.bn2      = nn.BatchNorm1d(hid)
        self.drop     = nn.Dropout(drop)
        self.res_proj = nn.Linear(in_ch, hid, bias=False)

    def forward(self, x, ei):
        h = F.elu(self.bn1(self.conv1(x, ei))) + self.res_proj(x)
        h = self.drop(h)
        h = F.elu(self.bn2(self.conv2(h, ei)))
        h = self.drop(h)
        return self.conv3(h, ei)


class BilinearDecoder(nn.Module):
    def __init__(self, emb):
        super().__init__()
        self.W = nn.Parameter(torch.eye(emb) + 0.01 * torch.randn(emb, emb))

    def forward(self, zi, zj):
        return torch.sigmoid((zi * (zj @ self.W.t())).sum(-1))


class GNNAutoencoder(nn.Module):
    def __init__(self, in_ch, hid=64, emb=32, drop=0.3):
        super().__init__()
        self.encoder      = GraphSAGEEncoder(in_ch, hid, emb, drop)
        self.edge_dec     = BilinearDecoder(emb)
        self.feat_decoder = nn.Sequential(
            nn.Linear(emb, hid), nn.ELU(), nn.Dropout(drop), nn.Linear(hid, in_ch)
        )

    def encode(self, x, ei):        return self.encoder(x, ei)
    def decode_edges(self, z, ei):  return self.edge_dec(z[ei[0]], z[ei[1]])
    def decode_feats(self, z):      return self.feat_decoder(z)
    def forward(self, x, ei):       return self.encode(x, ei)


# ── Training loop (returns best embedding Z and AUC) ──────────────────────
def train_gnn(pyg, epochs=400, patience=50, lr=2e-3,
              hid=64, emb=32, neg_ratio=3, alpha_feat=0.4):
    model = GNNAutoencoder(pyg.x.shape[1], hid, emb).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    def lr_lambda(ep):
        w = 50
        if ep < w: return (ep + 1) / w
        return 0.5 * (1 + np.cos(np.pi * (ep - w) / (epochs - w)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    ei   = pyg.edge_index
    npos = ei.shape[1]
    best_loss, best_state, no_imp = float("inf"), None, 0

    model.train()
    for ep in range(1, epochs + 1):
        opt.zero_grad()
        z    = model(pyg.x, ei)
        ps   = model.decode_edges(z, ei)
        nei  = negative_sampling(ei, pyg.num_nodes, npos * neg_ratio, method="sparse")
        ns   = model.decode_edges(z, nei)
        loss = F.binary_cross_entropy(
            torch.cat([ps, ns]),
            torch.cat([torch.ones(npos, device=DEVICE), torch.zeros(ns.shape[0], device=DEVICE)])
        ) + alpha_feat * F.mse_loss(model.decode_feats(z), pyg.x)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        v = loss.item()
        if v < best_loss - 1e-5:
            best_loss  = v
            best_state = {k: w.clone() for k, w in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
        if no_imp >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        Z    = model.encode(pyg.x, ei).cpu().numpy()
        ps_e = model.decode_edges(model.encode(pyg.x, ei), ei).cpu().numpy()
        nei2 = negative_sampling(ei, pyg.num_nodes, npos, method="sparse")
        ns_e = model.decode_edges(model.encode(pyg.x, ei), nei2).cpu().numpy()
    auc = roc_auc_score(
        np.array([1]*len(ps_e) + [0]*len(ns_e)),
        np.concatenate([ps_e, ns_e])
    )
    return Z, best_loss, auc


# ── Role assignment from embedding clusters ────────────────────────────────
def assign_roles(labels, X_raw, k=4):
    """Map cluster IDs → role names using mean raw degree + betweenness."""
    centers = np.array([X_raw[labels == c].mean(axis=0) for c in range(k)])
    md, mb  = np.median(centers[:, 0]), np.median(centers[:, 2])
    role_map = {}
    for c in range(k):
        d, b = centers[c, 0], centers[c, 2]
        if   d > md and b > mb: role_map[c] = "Keystone"
        elif d > md:             role_map[c] = "Generalist"
        elif b > mb:             role_map[c] = "Connector"
        else:                    role_map[c] = "Specialist"
    # guarantee uniqueness by priority
    if len(set(role_map.values())) < k:
        priority = ["Generalist", "Keystone", "Connector", "Specialist"]
        for rank, c in enumerate(np.argsort(centers[:, 0])[::-1]):
            role_map[c] = priority[rank]
    return role_map


print("All helpers defined OK")

### Cell 3 — Main loop: process all 176 networks

In [ ]:
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "M_PL_*.csv")))
print(f"Found {len(csv_files)} CSV files\n")

results   = []   # one dict per network
failed    = []   # networks that errored

t0 = time.time()

for idx, csv_path in enumerate(csv_files):
    net_id = os.path.splitext(os.path.basename(csv_path))[0]

    try:
        # ── 1. Load matrix ──────────────────────────────────────────────
        df = pd.read_csv(csv_path, index_col=0).fillna(0).astype(float)
        n_pl, n_po = df.shape[0], df.shape[1]
        n_edges    = int((df > 0).sum().sum())

        # Skip trivially tiny networks
        if n_pl < 3 or n_po < 3 or n_edges < 5:
            print(f"  [{idx+1:3d}] SKIP {net_id}  (too small: {n_pl}p × {n_po}q, {n_edges}e)")
            continue

        connectance = n_edges / (n_pl * n_po)

        # ── 2. Build graph + features ────────────────────────────────────
        G = build_bipartite(df)
        pyg, node_order, X_raw = build_pyg_data(G)
        pyg = pyg.to(DEVICE)

        # ── 3. Train GNN ─────────────────────────────────────────────────
        Z, best_loss, auc = train_gnn(pyg)

        # ── 4. Cluster embeddings ─────────────────────────────────────────
        # Silhouette sweep to find best k (2..min(6, N-1))
        max_k    = min(6, len(node_order) - 1)
        best_k, best_sil = 4, -1
        for k in range(2, max_k + 1):
            km  = KMeans(n_clusters=k, random_state=SEED, n_init=20)
            lbl = km.fit_predict(Z)
            s   = silhouette_score(Z, lbl)
            if s > best_sil:
                best_sil, best_k = s, k

        # Final clustering with k=4 for comparable roles
        km4     = KMeans(n_clusters=4, random_state=SEED, n_init=50)
        labels4 = km4.fit_predict(Z)
        sil4    = silhouette_score(Z, labels4)

        role_map = assign_roles(labels4, X_raw)
        roles    = [role_map[l] for l in labels4]

        # ── 5. Per-node CSV ───────────────────────────────────────────────
        Z2 = PCA(n_components=2, random_state=SEED).fit_transform(Z)
        node_df = pd.DataFrame({
            "species"    : node_order,
            "kind"       : [G.nodes[n].get("kind", "?") for n in node_order],
            "gnn_cluster": labels4,
            "gnn_role"   : roles,
            "pca1"       : Z2[:, 0],
            "pca2"       : Z2[:, 1],
        })
        node_df.to_csv(
            os.path.join(OUT_DIR, "per_network", f"{net_id}_roles.csv"),
            index=False
        )

        # ── 6. Network-level summary ──────────────────────────────────────
        role_series  = pd.Series(roles)
        poll_mask    = [G.nodes[n].get("kind") == "pollinator" for n in node_order]
        poll_roles   = [r for r, m in zip(roles, poll_mask) if m]
        poll_counts  = {r: poll_roles.count(r) for r in ALL_ROLES}

        # Dominant role = most frequent role among pollinators
        dominant = max(poll_counts, key=poll_counts.get)

        results.append({
            "network_id"  : net_id,
            "n_plants"    : n_pl,
            "n_pollinators": n_po,
            "n_edges"     : n_edges,
            "connectance" : round(connectance, 4),
            "best_loss"   : round(best_loss, 4),
            "auc"         : round(auc, 4),
            "silhouette_k4": round(sil4, 4),
            "best_k"      : best_k,
            "dominant_role": dominant,
            **{f"pct_{r}": round(poll_counts[r] / max(n_po, 1) * 100, 2)
               for r in ALL_ROLES},
            **{f"n_{r}": poll_counts[r] for r in ALL_ROLES},
        })

        elapsed = time.time() - t0
        print(f"  [{idx+1:3d}] OK  {net_id:<22}  "
              f"{n_pl:3d}p×{n_po:3d}q  "
              f"auc={auc:.3f}  sil={sil4:.3f}  "
              f"dom={dominant}  [{elapsed:.0f}s]")

    except Exception as e:
        failed.append((net_id, str(e)))
        print(f"  [{idx+1:3d}] ERR {net_id}: {e}")


# ── Save master summary ───────────────────────────────────────────────────
summary_df = pd.DataFrame(results)
master_csv = os.path.join(OUT_DIR, "all_networks_gnn_summary.csv")
summary_df.to_csv(master_csv, index=False)

print(f"\n{'='*60}")
print(f"Processed : {len(results)} / {len(csv_files)} networks")
print(f"Failed    : {len(failed)}")
print(f"Saved     : {master_csv}")
print(f"Total time: {(time.time()-t0)/60:.1f} min")
if failed:
    for nid, err in failed:
        print(f"  FAIL {nid}: {err}")

### Cell 4 — Load results (run this if re-starting kernel after the loop)

In [ ]:
master_csv = os.path.join(OUT_DIR, "all_networks_gnn_summary.csv")
summary_df = pd.read_csv(master_csv)
print(f"Loaded {len(summary_df)} network summaries")
print(summary_df.describe().round(3))

### Cell 5 — Visualisation 1: Role distribution heatmap across all networks

In [ ]:
"""
Viz 1 — Heatmap: each row = one network, columns = % Generalist/Specialist/Connector/Keystone
Networks sorted by dominant role then by network size.
This is the signature visualisation — shows at a glance which ecosystems
are dominated by which functional roles.
"""
pct_cols = [f"pct_{r}" for r in ALL_ROLES]

# Sort: dominant role group, then by size
sort_order = {"Generalist": 0, "Keystone": 1, "Connector": 2, "Specialist": 3}
df_plot = summary_df.copy()
df_plot["dom_order"] = df_plot["dominant_role"].map(sort_order)
df_plot = df_plot.sort_values(["dom_order", "n_pollinators"], ascending=[True, False])

heat_data = df_plot[pct_cols].values          # shape [N_networks, 4]
net_labels = df_plot["network_id"].str.replace("M_PL_", "").values

fig, ax = plt.subplots(figsize=(7, max(8, len(df_plot) * 0.18)), facecolor=BG)
style_ax(ax, "Functional Role Distribution — All Pollination Networks")

im = ax.imshow(heat_data, aspect="auto", cmap="RdYlGn",
               vmin=0, vmax=100, interpolation="nearest")
ax.set_xticks(range(4))
ax.set_xticklabels(ALL_ROLES, color=TXT, fontsize=8)
ax.set_yticks(range(len(df_plot)))
ax.set_yticklabels(net_labels, fontsize=5, color=TXT)

# Colour-coded left tick labels by dominant role
for tick, label in zip(ax.get_yticklabels(), df_plot["dominant_role"]):
    tick.set_color(ROLE_COLORS[label])

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.ax.set_ylabel("% of pollinators", color=TXT, fontsize=7)
cbar.ax.tick_params(colors=TXT, labelsize=6)

# Dominant role divider lines
prev = None
for i, dom in enumerate(df_plot["dominant_role"].values):
    if dom != prev and prev is not None:
        ax.axhline(i - 0.5, color=TXT, lw=0.6, alpha=0.4)
    prev = dom

plt.tight_layout()
path1 = os.path.join(OUT_DIR, "viz1_role_heatmap.png")
fig.savefig(path1, dpi=180, bbox_inches="tight", facecolor=BG)
plt.close()
print(f"Saved → {path1}")

: 

### Cell 6 — Visualisation 2: Role proportion stacked bar (sorted by % Generalist)

In [ ]:
"""
Viz 2 — Stacked horizontal bar chart.
Each bar = one network, split into 4 role proportions.
Sorted by % Generalist descending — reveals which networks have the richest
functional diversity.
"""
df_bar = summary_df.sort_values("pct_Generalist", ascending=True).reset_index(drop=True)
n      = len(df_bar)
y      = np.arange(n)
bar_h  = max(0.35, min(0.85, 12 / n))

fig, ax = plt.subplots(figsize=(10, max(6, n * 0.22)), facecolor=BG)
style_ax(ax, "Pollinator Role Composition per Network (sorted by % Generalist)")

left = np.zeros(n)
for role in ALL_ROLES:
    vals = df_bar[f"pct_{role}"].values
    ax.barh(y, vals, left=left, height=bar_h,
            color=ROLE_COLORS[role], alpha=0.88, label=role)
    left += vals

ax.set_yticks(y)
ax.set_yticklabels(
    df_bar["network_id"].str.replace("M_PL_", ""),
    fontsize=5.5, color=TXT
)
ax.set_xlabel("% of pollinators")
ax.set_xlim(0, 101)
ax.grid(axis="x", color=GRID, ls="--", lw=0.4)
legend_els = [mpatches.Patch(color=ROLE_COLORS[r], label=r) for r in ALL_ROLES]
ax.legend(handles=legend_els, loc="lower right",
          fontsize=8, facecolor="#0D1117", labelcolor=TXT)

plt.tight_layout()
path2 = os.path.join(OUT_DIR, "viz2_stacked_bars.png")
fig.savefig(path2, dpi=180, bbox_inches="tight", facecolor=BG)
plt.close()
print(f"Saved → {path2}")

### Cell 7 — Visualisation 3: Connectance vs role proportions (scatter matrix)

In [ ]:
"""
Viz 3 — 2×2 scatter grid: connectance (x-axis) vs % each role (y-axis).
Points coloured by dominant role.  Trend line added.
Tests the ecological hypothesis: denser networks → more generalists.
"""
from scipy.stats import pearsonr

fig, axes = plt.subplots(2, 2, figsize=(11, 8), facecolor=BG)
fig.suptitle("Network Connectance vs GNN-Discovered Role Proportions",
             color=TXT, fontsize=11, fontweight="bold")

for ax, role in zip(axes.flat, ALL_ROLES):
    style_ax(ax, f"Connectance vs % {role}")
    x = summary_df["connectance"].values
    y = summary_df[f"pct_{role}"].values
    c = [ROLE_COLORS[r] for r in summary_df["dominant_role"]]
    ax.scatter(x, y, c=c, s=25, alpha=0.75, linewidths=0)

    # Trend line
    z      = np.polyfit(x, y, 1)
    xi     = np.linspace(x.min(), x.max(), 100)
    ax.plot(xi, np.polyval(z, xi), "--", color="white", lw=1, alpha=0.6)

    r, p = pearsonr(x, y)
    ax.set_xlabel("Connectance")
    ax.set_ylabel(f"% {role}")
    ax.text(0.97, 0.95, f"r={r:.2f}  p={p:.3f}",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=7.5, color=TXT)
    ax.grid(color=GRID, ls="--", lw=0.4)

# Shared legend
legend_els = [mpatches.Patch(color=ROLE_COLORS[r], label=f"dom. {r}") for r in ALL_ROLES]
fig.legend(handles=legend_els, loc="lower center", ncol=4,
           fontsize=8, facecolor="#0D1117", labelcolor=TXT,
           bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
path3 = os.path.join(OUT_DIR, "viz3_connectance_scatter.png")
fig.savefig(path3, dpi=180, bbox_inches="tight", facecolor=BG)
plt.close()
print(f"Saved → {path3}")

### Cell 8 — Visualisation 4: AUC & Silhouette quality distributions

In [ ]:
"""
Viz 4 — Quality metrics across all 176 runs.
Left : histogram of AUC scores   (confirms GNN learned real structure)
Right: histogram of Silhouette scores  (confirms clusters are valid)
This is your 'we ran it on 176 networks and it works' evidence panel.
"""
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), facecolor=BG)
fig.suptitle("GNN Quality Metrics Across All Networks",
             color=TXT, fontsize=11, fontweight="bold")

# AUC histogram
style_ax(ax1, "Link-Prediction AUC Distribution")
ax1.hist(summary_df["auc"], bins=25, color="#42A5F5", alpha=0.85, edgecolor=BG)
ax1.axvline(summary_df["auc"].mean(), color="#FFD700", ls="--", lw=1.5,
            label=f'mean={summary_df["auc"].mean():.3f}')
ax1.axvline(0.85, color="#FF7043", ls=":", lw=1.2, label="threshold 0.85")
ax1.set_xlabel("AUC")
ax1.set_ylabel("Number of networks")
ax1.legend(fontsize=8, facecolor=BG, labelcolor=TXT)
ax1.grid(color=GRID, ls="--", lw=0.4)
good_auc = (summary_df["auc"] >= 0.85).sum()
ax1.text(0.03, 0.92, f"{good_auc}/{len(summary_df)} networks AUC≥0.85",
         transform=ax1.transAxes, color=TXT, fontsize=8)

# Silhouette histogram
style_ax(ax2, "Clustering Silhouette Score Distribution (k=4)")
ax2.hist(summary_df["silhouette_k4"], bins=25, color="#4CAF50", alpha=0.85, edgecolor=BG)
ax2.axvline(summary_df["silhouette_k4"].mean(), color="#FFD700", ls="--", lw=1.5,
            label=f'mean={summary_df["silhouette_k4"].mean():.3f}')
ax2.axvline(0.25, color="#FF7043", ls=":", lw=1.2, label="acceptable 0.25")
ax2.set_xlabel("Silhouette score")
ax2.set_ylabel("Number of networks")
ax2.legend(fontsize=8, facecolor=BG, labelcolor=TXT)
ax2.grid(color=GRID, ls="--", lw=0.4)

plt.tight_layout()
path4 = os.path.join(OUT_DIR, "viz4_quality_distributions.png")
fig.savefig(path4, dpi=180, bbox_inches="tight", facecolor=BG)
plt.close()
print(f"Saved → {path4}")

### Cell 9 — Visualisation 5: Network size vs role diversity (bubble chart)

In [ ]:
"""
Viz 5 — Bubble chart.
x = n_plants,  y = n_pollinators,  bubble size = n_edges,
bubble colour = dominant role.
Answers: do larger networks have more diverse role structures?
"""
fig, ax = plt.subplots(figsize=(10, 7), facecolor=BG)
style_ax(ax, "Network Size vs Pollinator Count — coloured by dominant GNN role")

for role in ALL_ROLES:
    sub = summary_df[summary_df["dominant_role"] == role]
    ax.scatter(
        sub["n_plants"], sub["n_pollinators"],
        s=np.sqrt(sub["n_edges"]) * 8,
        c=ROLE_COLORS[role], alpha=0.75, linewidths=0.4,
        edgecolors="white", label=role
    )

ax.set_xlabel("Number of plant species")
ax.set_ylabel("Number of pollinator species")
ax.grid(color=GRID, ls="--", lw=0.4)

# Size legend
for ne, lab in [(50, "50 edges"), (200, "200 edges"), (500, "500 edges")]:
    ax.scatter([], [], s=np.sqrt(ne)*8, c="grey", alpha=0.6, label=lab)

ax.legend(fontsize=8, facecolor=BG, labelcolor=TXT,
          loc="upper left", framealpha=0.6)

plt.tight_layout()
path5 = os.path.join(OUT_DIR, "viz5_size_bubble.png")
fig.savefig(path5, dpi=180, bbox_inches="tight", facecolor=BG)
plt.close()
print(f"Saved → {path5}")

### Cell 10 — Visualisation 6: Cross-network role frequency (summary panel)

In [ ]:
"""
Viz 6 — 2×2 summary panel (the 'thesis figure').
  Top-left  : donut — dominant role frequency across all networks
  Top-right : box plot — % each role across all networks
  Bot-left  : scatter — silhouette vs AUC (quality correlation)
  Bot-right : bar — mean role % ranked by size bin (small/medium/large)
"""
fig = plt.figure(figsize=(14, 10), facecolor=BG)
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# ── Top-left: dominant role donut ─────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
style_ax(ax1, "Dominant Role Frequency (N networks)")
dom_counts = summary_df["dominant_role"].value_counts()
wedge_cols = [ROLE_COLORS[r] for r in dom_counts.index]
wedges, texts, autotexts = ax1.pie(
    dom_counts.values,
    labels=dom_counts.index,
    colors=wedge_cols,
    autopct="%1.0f%%",
    startangle=90,
    pctdistance=0.78,
    wedgeprops={"width": 0.55, "edgecolor": BG, "linewidth": 1.5}
)
for t in texts + autotexts:
    t.set_color(TXT)
    t.set_fontsize(8.5)
ax1.text(0, 0, f"n={len(summary_df)}", ha="center", va="center",
         color=TXT, fontsize=12, fontweight="bold")

# ── Top-right: box plot per role ──────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
style_ax(ax2, "Role % Distribution Across All Networks")
box_data   = [summary_df[f"pct_{r}"].values for r in ALL_ROLES]
bp         = ax2.boxplot(box_data, patch_artist=True, widths=0.55,
                          medianprops={"color": "white", "lw": 2})
for patch, role in zip(bp["boxes"], ALL_ROLES):
    patch.set_facecolor(ROLE_COLORS[role])
    patch.set_alpha(0.8)
for element in ["whiskers", "caps", "fliers"]:
    for item in bp[element]:
        item.set_color(TXT)
        item.set_alpha(0.5)
ax2.set_xticks(range(1, 5))
ax2.set_xticklabels(ALL_ROLES, color=TXT, fontsize=8)
ax2.set_ylabel("% of pollinators")
ax2.grid(axis="y", color=GRID, ls="--", lw=0.4)

# ── Bot-left: silhouette vs AUC ───────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
style_ax(ax3, "GNN Quality: Silhouette vs AUC")
for role in ALL_ROLES:
    sub = summary_df[summary_df["dominant_role"] == role]
    ax3.scatter(sub["auc"], sub["silhouette_k4"],
                c=ROLE_COLORS[role], s=22, alpha=0.75,
                linewidths=0, label=role)
ax3.axvline(0.85, color="#FF7043", ls=":", lw=0.8, alpha=0.7)
ax3.axhline(0.25, color="#FF7043", ls=":", lw=0.8, alpha=0.7)
ax3.set_xlabel("Link-prediction AUC")
ax3.set_ylabel("Silhouette (k=4)")
ax3.legend(fontsize=7, facecolor=BG, labelcolor=TXT)
ax3.grid(color=GRID, ls="--", lw=0.4)

# ── Bot-right: mean role % by size bin ───────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
style_ax(ax4, "Mean Role % by Network Size")

summary_df["size_bin"] = pd.qcut(
    summary_df["n_pollinators"], q=3,
    labels=["Small", "Medium", "Large"]
)
grouped = summary_df.groupby("size_bin", observed=True)[
    [f"pct_{r}" for r in ALL_ROLES]
].mean()

x      = np.arange(3)
width  = 0.18
for i, role in enumerate(ALL_ROLES):
    ax4.bar(x + i * width, grouped[f"pct_{role}"].values,
            width, label=role, color=ROLE_COLORS[role], alpha=0.85)

ax4.set_xticks(x + width * 1.5)
ax4.set_xticklabels(["Small", "Medium", "Large"], color=TXT, fontsize=9)
ax4.set_ylabel("Mean % pollinators")
ax4.legend(fontsize=7.5, facecolor=BG, labelcolor=TXT)
ax4.grid(axis="y", color=GRID, ls="--", lw=0.4)

fig.suptitle(
    f"GNN Functional Role Analysis — {len(summary_df)} Pollination Networks (Web of Life)",
    color=TXT, fontsize=12, fontweight="bold", y=1.01
)
path6 = os.path.join(OUT_DIR, "viz6_summary_panel.png")
fig.savefig(path6, dpi=180, bbox_inches="tight", facecolor=BG)
plt.close()
print(f"Saved → {path6}")

### Cell 11 — Print final cross-network summary

In [ ]:
print("\n" + "="*65)
print("CROSS-NETWORK GNN SUMMARY")
print("="*65)
print(f"  Networks processed          : {len(summary_df)}")
print(f"  Mean AUC                    : {summary_df['auc'].mean():.4f} ± {summary_df['auc'].std():.4f}")
print(f"  Networks with AUC ≥ 0.85   : {(summary_df['auc']>=0.85).sum()}")
print(f"  Mean Silhouette (k=4)       : {summary_df['silhouette_k4'].mean():.4f} ± {summary_df['silhouette_k4'].std():.4f}")
print("-"*65)
print("  Mean pollinator role proportions:")
for r in ALL_ROLES:
    m = summary_df[f'pct_{r}'].mean()
    s = summary_df[f'pct_{r}'].std()
    print(f"    {r:<14} {m:5.1f}% ± {s:4.1f}%")
print("-"*65)
print("  Dominant role frequency:")
for r, c in summary_df["dominant_role"].value_counts().items():
    pct = 100 * c / len(summary_df)
    print(f"    {r:<14} {c:3d} networks ({pct:.1f}%)")
print("="*65)
print(f"\nAll outputs saved in:\n  {OUT_DIR}")